# 00 - Introduction to Retrieval-Augmented Generation (RAG)

## 🎯 Learning Objectives

By the end of this notebook, you will:
- Understand what RAG is and why it's important
- Learn about different RAG architectures (2-Step, Agentic, Hybrid)
- Set up SiliconFlow models using langchain-dev-utils
- Test connections to OceanBase and SiliconFlow

## 📚 What is RAG?

Large Language Models (LLMs) are powerful, but they have two key limitations:

1. **Finite context** — they can't ingest entire corpora at once
2. **Static knowledge** — their training data is frozen at a point in time

**Retrieval-Augmented Generation (RAG)** addresses these problems by:
- Fetching relevant external knowledge at query time
- Combining retrieval with generation to produce grounded, context-aware answers

### RAG Architectures

| Architecture    | Description                                                 | Control | Flexibility | Latency    |
| --------------- | ----------------------------------------------------------- | ------- | ----------- | ---------- |
| **2-Step RAG**  | Retrieval always happens before generation                 | ✅ High  | ❌ Low       | ⚡ Fast     |
| **Agentic RAG** | LLM-powered agent decides when and how to retrieve         | ❌ Low   | ✅ High      | ⏳ Variable |
| **Hybrid RAG**  | Combines both approaches with validation steps             | ⚖️ Medium | ⚖️ Medium   | ⏳ Variable |

## 🛠️ Environment Setup

### Prerequisites

1. **Python 3.11+** installed (required by langchain-dev-utils)
2. **OceanBase** installed and running ([Local installation guide](https://www.oceanbase.com/docs/community-observer-cn))
3. **SiliconFlow API Key** from [cloud.siliconflow.cn](https://cloud.siliconflow.cn)
4. **Environment variables** configured in `.env` file

### OceanBase Dashboard (SeekDB)

Once OceanBase is running, you can access the SeekDB dashboard to monitor your database:

![SeekDB Desktop Dashboard](../shared/public/zh/seekdb-desktop.png)

The dashboard shows:
- Database status and version
- Connection details (host, port, user)
- Active databases and tables
- System monitoring and metrics

### Install uv (Fast Python Package Manager)

**macOS & Linux:**
```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

**Windows (PowerShell):**
```powershell
powershell -c "irm https://astral.sh/uv/install.ps1 | iex"
```

**Alternative (using pip):**
```bash
pip install uv
```

**Verify installation:**
```bash
uv --version
```

### Install Dependencies

```bash
# From the workspace root, install all dependencies (Python 3.11+ required)
uv sync --dev

# Copy and configure environment variables
cp .env.example .env
# Edit .env and add your API keys:
#   - SILICONFLOW_API_KEY (get from https://cloud.siliconflow.cn)
#   - OCEANBASE_* connection parameters
#   - LANGCHAIN_API_KEY (optional, for LangSmith tracing)
```

### Start Jupyter

```bash
# Start Jupyter Lab
jupyter lab packages/ep2-langchain-retrieval/notebooks/

# Or Jupyter Notebook
jupyter notebook packages/ep2-langchain-retrieval/notebooks/
```

### Load Environment Variables

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")

# Verify required environment variables
required_vars = [
    "SILICONFLOW_API_KEY",
    "OCEANBASE_HOST",
    "OCEANBASE_PORT",
    "OCEANBASE_USER",
    "OCEANBASE_DB",
]

missing_vars = [var for var in required_vars if not os.getenv(var)]

if missing_vars:
    print("❌ Missing environment variables:")
    for var in missing_vars:
        print(f"   - {var}")
    print("\nPlease check your .env file!")
    raise ValueError("Missing required environment variables")
else:
    print("✅ All environment variables loaded successfully!")
    print(f"\n📍 OceanBase: {os.getenv('OCEANBASE_HOST')}:{os.getenv('OCEANBASE_PORT')}")
    print(f"📍 Database: {os.getenv('OCEANBASE_DB')}")
    print(f"📍 Chat Model: {os.getenv('SILICONFLOW_CHAT_MODEL', 'Qwen/Qwen2.5-7B-Instruct')}")
    print(f"📍 Embedding Model: {os.getenv('SILICONFLOW_EMBEDDING_MODEL', 'BAAI/bge-m3')}")

## 🤖 Register SiliconFlow as OpenAI-Compatible Provider

SiliconFlow provides an OpenAI-compatible API. We'll use `langchain-dev-utils` to register it as a provider.

In [2]:
from langchain_dev_utils.chat_models import register_model_provider
from langchain_dev_utils.embeddings import register_embeddings_provider

# SiliconFlow configuration
SILICONFLOW_BASE_URL = os.getenv("SILICONFLOW_BASE_URL", "https://api.siliconflow.cn/v1")

# Register SiliconFlow chat model provider
register_model_provider(
    provider_name="siliconflow",
    chat_model="openai-compatible",  # SiliconFlow uses OpenAI-compatible API
    base_url=SILICONFLOW_BASE_URL,
)

# Register SiliconFlow embeddings provider
register_embeddings_provider(
    provider_name="siliconflow",  # Fixed: use provider_name instead of provider
    embeddings_model="openai-compatible",
    base_url=SILICONFLOW_BASE_URL,
)

print("✅ SiliconFlow registered as OpenAI-compatible provider!")
print(f"🌐 Base URL: {SILICONFLOW_BASE_URL}")

✅ SiliconFlow registered as OpenAI-compatible provider!
🌐 Base URL: https://api.siliconflow.cn/v1


## 💬 Test Chat Model

Let's load and test the Qwen chat model from SiliconFlow.

In [3]:
from langchain_dev_utils.chat_models import load_chat_model
from langchain_core.messages import HumanMessage

# Load chat model using provider:model format
CHAT_MODEL_NAME = os.getenv("SILICONFLOW_CHAT_MODEL", "Qwen/Qwen2.5-7B-Instruct")

try:
    chat_model = load_chat_model(f"siliconflow:{CHAT_MODEL_NAME}")
    
    # Test with a simple message
    response = chat_model.invoke([
        HumanMessage(content="Hello! Please introduce yourself in one sentence.")
    ])
    
    print("✅ Chat model test successful!")
    print(f"\n🤖 Model: {CHAT_MODEL_NAME}")
    print(f"💬 Response: {response.content}")
    
except Exception as e:
    print("❌ Chat model test failed!")
    print(f"Error: {e}")
    raise

✅ Chat model test successful!

🤖 Model: Qwen/Qwen2.5-7B-Instruct
💬 Response: Hello! I'm Qwen, a large language model created by Alibaba Cloud, here to assist you with any questions or conversations you'd like to have.


## 🧬 Test Embedding Model

Now let's test the BGE-M3 embedding model from SiliconFlow.

In [ ]:
from langchain_dev_utils.embeddings import load_embeddings

# Load embedding model using provider:model format
EMBEDDING_MODEL_NAME = os.getenv("SILICONFLOW_EMBEDDING_MODEL", "BAAI/bge-m3")

try:
    embeddings = load_embeddings(f"siliconflow:{EMBEDDING_MODEL_NAME}")
    
    # Test embedding generation
    test_text = "This is a test sentence for embedding."
    embedding_vector = embeddings.embed_query(test_text)
    
    print("✅ Embedding model test successful!")
    print(f"\n🧬 Model: {EMBEDDING_MODEL_NAME}")
    print(f"📏 Embedding dimension: {len(embedding_vector)}")
    print(f"🔢 First 5 values: {embedding_vector[:5]}")
    
    # Store for later use
    print(f"\n💾 Embedding model loaded and ready to use!")
    
except Exception as e:
    print("❌ Embedding model test failed!")
    print(f"Error: {e}")
    raise

✅ Embedding model test successful!

🧬 Model: BAAI/bge-m3
📏 Embedding dimension: 1024
🔢 First 5 values: [-0.04303452745079994, 0.024637650698423386, -0.016472959890961647, -0.015171202830970287, -0.0023055921774357557]

💾 Embedding model loaded and ready to use!


## 🗄️ Test OceanBase Vector Store

Let's test the connection to OceanBase and create a vector store using our real SiliconFlow embeddings.

In [14]:
import os

from langchain_oceanbase.vectorstores import OceanbaseVectorStore
from langchain_core.documents import Document

# OceanBase connection parameters
connection_args = {
    "host": os.getenv("OCEANBASE_HOST", "127.0.0.1"),
    "port": int(os.getenv("OCEANBASE_PORT", "2881")),
    "user": os.getenv("OCEANBASE_USER", "root"),
    "password": os.getenv("OCEANBASE_PASSWORD", ""),
    "db_name": os.getenv("OCEANBASE_DB", "test"),
}

try:
    # Create vector store with SiliconFlow embeddings
    vector_store = OceanbaseVectorStore(
        embedding_function=embeddings,  # Using real SiliconFlow embeddings
        table_name="test_connection",
        connection_args=connection_args,
        drop_old=True,  # Clean slate for testing
    )
    
    # Add test documents
    test_docs = [
        Document(
            page_content="LangChain is a framework for developing applications powered by language models.",
            metadata={"source": "test", "topic": "LangChain"}
        ),
        Document(
            page_content="OceanBase is a distributed relational database with vector search capabilities.",
            metadata={"source": "test", "topic": "OceanBase"}
        )
    ]
    
    ids = vector_store.add_documents(test_docs)
    
    print("✅ OceanBase connection successful!")
    print(f"📊 Connected to: {connection_args['host']}:{connection_args['port']}")
    print(f"🗄️ Database: {connection_args['db_name']}")
    print(f"📝 Added {len(ids)} test documents")
    print(f"🆔 Document IDs: {ids}")
    
    # Test similarity search
    results = vector_store.similarity_search("database", k=1)
    print(f"\n🔍 Similarity search test:")
    print(f"   Query: 'database'")
    print(f"   Result: {results[0].page_content}")
    
    # Cleanup: Delete test documents
    print(f"\n🧹 Cleaning up test documents...")
    vector_store.delete(ids)
    print(f"✅ Deleted {len(ids)} test documents")
    
    # Verify deletion
    remaining = vector_store.similarity_search("database", k=10)
    print(f"📊 Remaining documents in test_connection table: {len(remaining)}")
    
except Exception as e:
    print("❌ OceanBase connection failed!")
    print(f"Error: {e}")
    print("\nPlease check:")
    print("  1. OceanBase is running")
    print("  2. Connection parameters in .env are correct")
    print("  3. Network connectivity")
    raise

✅ OceanBase connection successful!
📊 Connected to: 127.0.0.1:2881
🗄️ Database: test
📝 Added 2 test documents
🆔 Document IDs: ['65a5d8c6-90ee-4125-be0d-944852108b0f', 'e8378432-2414-456c-8dac-e412133a9b8d']

🔍 Similarity search test:
   Query: 'database'
   Result: OceanBase is a distributed relational database with vector search capabilities.

🧹 Cleaning up test documents...
✅ Deleted 2 test documents
📊 Remaining documents in test_connection table: 0


## 🎉 Summary

Congratulations! If all tests passed, you have successfully:

- ✅ Understood what RAG is and its importance
- ✅ Learned about different RAG architectures
- ✅ Configured SiliconFlow as an OpenAI-compatible provider
- ✅ Tested Qwen/Qwen2.5-7B-Instruct chat model
- ✅ Tested BAAI/bge-m3 embedding model
- ✅ Connected to OceanBase vector store
- ✅ Performed basic vector similarity search

### Next Steps:

In **Notebook 01**, we'll build a comprehensive knowledge base by:
- Loading documents from various sources (PDF, web, text)
- Splitting text into optimal chunks
- Creating embeddings with BGE-M3
- Storing vectors in OceanBase with metadata
- Performing advanced similarity searches

## 🧹 Cleanup Test Data

Let's clean up the test data we created to keep the database tidy.

In [ ]:
# Delete all documents from the test_connection table
print("🧹 Cleaning up all test data from OceanBase...")

try:
    # Use filter to delete all documents (WHERE 1=1 means all rows)
    vector_store.delete(fltr="1=1")
    print("✅ Successfully deleted all documents from test_connection table")
    
    # Verify the table is empty
    remaining = vector_store.similarity_search("", k=100)
    print(f"📊 Remaining documents: {len(remaining)}")
    
    if len(remaining) == 0:
        print("✨ Database is clean and ready for Notebook 01!")
    
except Exception as e:
    print(f"⚠️  Cleanup warning: {e}")
    print("💡 This is normal if the table is already empty or doesn't exist yet.")

## 💡 Additional Resources

- [LangChain RAG Documentation](https://python.langchain.com/docs/tutorials/rag/)
- [OceanBase LangChain Integration](https://github.com/oceanbase/langchain-oceanbase)
- [SiliconFlow API Documentation](https://docs.siliconflow.cn)
- [LangChain Dev Utils](https://tbice123123.github.io/langchain-dev-utils-docs/)
- [BGE-M3 Embedding Model](https://huggingface.co/BAAI/bge-m3)
- [Qwen2.5 Model](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)